# Experiment 06: Implement Decision Tree and Random Forest Models in Python

**Aim:** To implement and compare Decision Tree and Random Forest classifiers on the Cleveland Heart Disease dataset, investigate the effect of tree depth on training vs testing generalization, and extract feature importances.

### Dataset Description
**Dataset:** Cleveland Heart Disease Clinical Dataset (297 complete patient records, 13 clinical risk factors).
- `age`: Patient age in years.
- `sex`: Gender (1 = male, 0 = female).
- `cp`: Chest pain type (1: typical angina, 2: atypical, 3: non-anginal, 4: asymptomatic).
- `trestbps`: Resting blood pressure (mm Hg).
- `chol`: Serum cholesterol (mg/dl).
- `fbs`: Fasting blood sugar > 120 mg/dl (1/0).
- `restecg`: Resting ECG results (0, 1, 2).
- `thalach`: Maximum heart rate achieved during exercise test.
- `exang`: Exercise-induced angina (1/0).
- `oldpeak`: ST depression induced by exercise relative to rest.
- `slope`: Slope of peak exercise ST segment.
- `ca`: Number of major vessels colored by fluoroscopy (0-3).
- `thal`: Thallium heart scan (3: normal, 6: fixed defect, 7: reversible defect).
- Target: Heart disease diagnosis (0: Normal [160], 1: Disease Present [137]).

### Theory
A Decision Tree greedily partitions feature space by maximizing Information Gain or minimizing Gini Impurity: Gini = 1 - sum(p_i^2). Unconstrained trees split until leaves are pure, causing catastrophic overfitting. Random Forest constructs an ensemble of B bootstrap-aggregated (bagged) trees, introducing feature subspace randomization (sqrt(p) features evaluated at each split). Ensemble voting drastically reduces estimator variance without increasing bias.

#### 1. Load Dataset, Clean Missing Values, and Stratified Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv('../datasets/cleveland_heart.csv', header=None, names=cols, na_values='?').dropna()
df['target'] = (df['target'] > 0).astype(int)
X, y = df.drop(columns='target'), df['target']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train Samples: {len(Xtr)}, Test Samples: {len(Xte)}')

#### 2. Train Decision Trees Across Depths to Demonstrate Overfitting

In [ ]:
depths = [2, 3, 4, 5, None]
train_acc, test_acc = [], []
for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xtr, ytr)
    train_acc.append(dt.score(Xtr, ytr))
    test_acc.append(dt.score(Xte, yte))
    print(f'Decision Tree (depth={str(d):4s}): Train Acc = {train_acc[-1]:.4f}, Test Acc = {test_acc[-1]:.4f}')

plt.figure(figsize=(8, 4))
plt.plot([2, 3, 4, 5, 6], train_acc, marker='o', label='Train Accuracy')
plt.plot([2, 3, 4, 5, 6], test_acc, marker='s', label='Test Accuracy')
plt.xticks([2, 3, 4, 5, 6], ['2', '3', '4', '5', 'Unlimited'])
plt.title('Decision Tree Overfitting: Train vs Test Accuracy across Max Depth')
plt.xlabel('Tree Max Depth')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

#### 3. Visualizing Optimal Pruned Decision Tree (Depth 3)

In [ ]:
best_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, ytr)
plt.figure(figsize=(16, 8))
plot_tree(best_tree, feature_names=X.columns, class_names=['Normal', 'Disease'], filled=True, rounded=True)
plt.title('Pruned Decision Tree (max_depth=3) for Heart Disease Diagnosis')
plt.show()

#### 4. Train Random Forest Classifier and Feature Importance Analysis

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)
print(f'Random Forest (100 trees): Train Acc = {rf.score(Xtr, ytr):.4f}, Test Acc = {rf.score(Xte, yte):.4f}')
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
plt.figure(figsize=(8, 6))
fi.plot(kind='barh', color='darkorange')
plt.title('Random Forest Gini Feature Importances')
plt.xlabel('Importance')
plt.show()

### Observations
1. The unconstrained Decision Tree (depth=None) achieved 100.0% training accuracy but suffered severe overfitting, falling to 70.00% on the test set.
2. Pruning the tree at max_depth=3 achieved the optimal single-tree test accuracy of 80.00%.
3. The 100-tree Random Forest outperformed all individual decision trees, reaching 85.00% test accuracy via ensemble variance reduction.
4. Gini feature importance identified chest pain type (cp: 0.1420), thallium defect (thal: 0.1216), maximum heart rate (thalach: 0.1213), and ST depression (oldpeak: 0.1146) as primary clinical drivers.

### Result
Random Forest ensembles demonstrate superior generalization (85.00%) compared to greedy single decision trees (80.00% pruned, 70.00% unconstrained). Hyperparameter depth constraints prevent overfitting in single trees, but bagging provides robust variance control.